<a href="https://colab.research.google.com/github/ighackerbot/3d_portfolio/blob/main/lion_dsp_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **`Install Required Audio Processing Libraries`**

In [3]:
# Install core robust audio signal manipulation tools
!pip install librosa soundfile

# **`Initialize Standardized Dataset Directory Structure`**

In [4]:
import os

# Establish strict taxonomic directory trees matching your proposal
directories = [
    "data/raw/panthera_leo",
    "data/raw/background_ambient",
    "data/processed/train/panthera_leo",
    "data/processed/train/background_ambient",
    "data/processed/validation/panthera_leo",
    "data/processed/validation/background_ambient",
    "data/processed/test/panthera_leo",
    "data/processed/test/background_ambient"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("✅ Professional folder architecture initialized successfully.")

✅ Professional folder architecture initialized successfully.


# **`Core DSP Preprocessing Pipeline`**

In [5]:
import os
import librosa
import soundfile as sf
import numpy as np

def run_dsp_pipeline(source_file_path, destination_folder):
    TARGET_SR = 32000
    SEGMENT_DURATION = 5
    SAMPLES_PER_SEGMENT = TARGET_SR * SEGMENT_DURATION

    try:
        # Load audio, auto-enforcing mono array conversion at 32kHz
        audio_signal, sr = librosa.load(source_file_path, sr=TARGET_SR, mono=True)

        peak_amplitude = np.max(np.abs(audio_signal))

        # Drop corrupted or silent data files to maintain vector integrity
        if peak_amplitude < 1e-6:
            print(f"Skipping silent file: {source_file_path}")
            return 0

        # Mathematical peak normalization to -3 dBFS
        target_peak = 10 ** (-3 / 20)
        audio_signal = (audio_signal / peak_amplitude) * target_peak

        base_name = os.path.splitext(os.path.basename(source_file_path))[0]
        total_samples = len(audio_signal)
        chunk_count = 0

        # Slice long timelines into uniform 5-second arrays (160,000 values)
        for start_sample in range(0, total_samples, SAMPLES_PER_SEGMENT):
            end_sample = start_sample + SAMPLES_PER_SEGMENT
            audio_chunk = audio_signal[start_sample:end_sample]

            # Zero-padding: pads short trailing clips with silent numbers
            if len(audio_chunk) < SAMPLES_PER_SEGMENT:
                audio_chunk = np.pad(
                    audio_chunk,
                    (0, SAMPLES_PER_SEGMENT - len(audio_chunk)),
                    mode='constant'
                )

            output_filename = f"{base_name}_chunk_{chunk_count}.wav"
            output_path = os.path.join(destination_folder, output_filename)

            # Export clean WAV segment
            sf.write(output_path, audio_chunk, TARGET_SR)
            chunk_count += 1

        return chunk_count

    except Exception as e:
        print(f"❌ Error processing {source_file_path}: {e}")
        return 0

# **`Automated Dataset Processing & Split Allocation`**

In [6]:
import glob
import random

random.seed(42)

def process_dataset():

    classes = {
        "panthera_leo":
        "/content/drive/MyDrive/Cymasonic Labs SBM project /MAMMALS/panthera_leo/**/*.*",

        "background_ambient":
        "/content/drive/MyDrive/Cymasonic Labs SBM project /background_ambient/*.*"
    }

    for class_name, pattern in classes.items():

        files = glob.glob(pattern, recursive=True)

        random.shuffle(files)

        total_files = len(files)

        print(f"\nFound {total_files} raw tracking files for class: {class_name}")

        for index, file_path in enumerate(files):

            ratio = index / total_files

            if ratio < 0.70:
                split = "train"
            elif ratio < 0.85:
                split = "validation"
            else:
                split = "test"

            destination = (
                f"/content/drive/MyDrive/PROCESSED_DATASET/{split}/{class_name}"
            )

            chunks = run_dsp_pipeline(
                file_path,
                destination
            )

            print(
                f" -> {os.path.basename(file_path)} "
                f"→ {split.upper()} "
                f"({chunks} chunks generated)"
            )

process_dataset()


Found 26 raw tracking files for class: panthera_leo
 -> Lion-Roar--Youtube-1.5sec.wav → TRAIN (1 chunks generated)
 -> lion-Roar-60sec.wav → TRAIN (14 chunks generated)
 -> Lion-Roar-2sec.wav → TRAIN (2 chunks generated)
 -> Lion-Moans---Youtube-90sec.wav → TRAIN (19 chunks generated)
 -> Lion-Roar-2sec (1).wav → TRAIN (1 chunks generated)
 -> Lion-Snarl-&-Growl--6sec.wav → TRAIN (2 chunks generated)
 -> Lion's-Growl---YouTube-25sec.wav → TRAIN (7 chunks generated)
 -> Lion-Roar-3sec.wav → TRAIN (2 chunks generated)
 -> Lion-Roar--Youtube-2sec.wav → TRAIN (1 chunks generated)
 -> Lion-(Panthera-leo)----xeno-canto-28sec.wav → TRAIN (7 chunks generated)
 -> Lion-Roar-4sec.wav → TRAIN (2 chunks generated)
 -> Lion-Roar--Youtube-90sec.wav → TRAIN (18 chunks generated)
 -> Lion-Growl-7sec.wav → TRAIN (2 chunks generated)
 -> Lion-Roar--Youtube-11sec.wav → TRAIN (3 chunks generated)
 -> Lion-loud-47sec.wav → TRAIN (12 chunks generated)
 -> Lion-Roar-,-Grunts-etc-205sec.wav → TRAIN (42 chunk

# `Check the data`

In [7]:
process_dataset()


Found 26 raw tracking files for class: panthera_leo
 -> Lion-Roar-3sec.wav → TRAIN (2 chunks generated)
 -> Lion---Panthera-leo---Macaulay-Library-50sec.wav → TRAIN (11 chunks generated)
 -> Lion-loud-47sec.wav → TRAIN (12 chunks generated)
 -> lion-grunts-sound-youtube-.wav → TRAIN (4 chunks generated)
 -> lion-Roar-60sec.wav → TRAIN (14 chunks generated)
 -> Lion-Roar--Youtube-11sec.wav → TRAIN (3 chunks generated)
 -> Lion-Roar-8sec..wav → TRAIN (3 chunks generated)
 -> Lion-Roar--Youtube-14sec.wav → TRAIN (3 chunks generated)
 -> Lion-Roar--Youtube-1.5sec.wav → TRAIN (1 chunks generated)
 -> Lion's-Growl---YouTube-25sec.wav → TRAIN (7 chunks generated)
 -> Lion-Roar--Youtube-2sec.wav → TRAIN (1 chunks generated)
 -> Lion-Growl-&-Snarl---Youtube-14sec.wav → TRAIN (3 chunks generated)
 -> Lion-(Panthera-leo)----xeno-canto-12sec.wav → TRAIN (4 chunks generated)
 -> Lion-Moans-Sound---YouTube-40sec.wav → TRAIN (9 chunks generated)
 -> Lion-Roar-,-Grunts-etc-205sec.wav → TRAIN (42 chun

# `Dataset Verification & Integrity **Check**`

In [8]:
import glob

print("--- STANDARDIZED DATASET INTEGRITY LOGS ---")

for split in ["train", "validation", "test"]:

    for cls in ["panthera_leo", "background_ambient"]:

        path = (
            f"/content/drive/MyDrive/PROCESSED_DATASET/"
            f"{split}/{cls}/*.wav"
        )

        count = len(glob.glob(path))

        print(
            f"Directory "
            f"[PROCESSED_DATASET/{split}/{cls}/] "
            f"contains: {count} verified WAV slices"
        )

--- STANDARDIZED DATASET INTEGRITY LOGS ---
Directory [PROCESSED_DATASET/train/panthera_leo/] contains: 191 verified WAV slices
Directory [PROCESSED_DATASET/train/background_ambient/] contains: 92 verified WAV slices
Directory [PROCESSED_DATASET/validation/panthera_leo/] contains: 54 verified WAV slices
Directory [PROCESSED_DATASET/validation/background_ambient/] contains: 33 verified WAV slices
Directory [PROCESSED_DATASET/test/panthera_leo/] contains: 32 verified WAV slices
Directory [PROCESSED_DATASET/test/background_ambient/] contains: 22 verified WAV slices


# **`Install Google Perch + ML Dependencies`**

In [9]:
# Install Perch ecosystem and ML libraries
!pip install -q "perch-hoplite[tf]"
!pip install -q librosa soundfile scikit-learn tqdm joblib pandas

# Core imports
import tensorflow as tf
import numpy as np
import librosa
import os
import glob
import pandas as pd

print("TensorFlow Version:", tf.__version__)

# Load Google Perch
from perch_hoplite.zoo import model_configs

print("Loading Google Perch v2 backbone...")
# Initialize frozen pretrained model
perch_model = model_configs.load_model_by_name("perch_v2")
print("✅ Google Perch v2 loaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━


100%|██████████| 305k/305k [00:00<00:00, 43.2MB/s]


  0%|          | 0.00/144k [00:00<?, ?B/s]

  0%|          | 0.00/176 [00:00<?, ?B/s]

100%|██████████| 176/176 [00:00<00:00, 20.4kB/s]

100%|██████████| 144k/144k [00:00<00:00, 3.70MB/s]

  0%|          | 0.00/176 [00:00<?, ?B/s]

100%|██████████| 176/176 [00:00<00:00, 453kB/s]




100%|██████████| 176/176 [00:00<00:00, 9.91kB/s]

100%|██████████| 9.02k/9.02k [00:00<00:00, 134kB/s]

...



  3%|▎         | 10.0M/388M [00:00<00:04, 95.1MB/s]

100%|██████████| 176/176 [00:00<00:00, 199kB/s]




  0%|          | 0.00/176 [00:00<?, ?B/s]

100%|██████████| 176/176 [00:00<00:00, 21.5kB/s]




  0%|          | 0.00/2.58M [00:00<?, ?B/s]




100%|██████████| 97.0/97.0 [00:00<00:00, 100kB/s]
100%|██████████| 2.58M/2.58M [00:00<00:00, 59.0MB/s]

  5%|▌         | 20.0M/388M [00:00<00:09, 39.5MB/s]
  7%|▋         | 26.0M/388M [00:00<00:08, 44.1MB/s]
 10%|█         | 39.0M/388M [00:00<00:05, 67.1MB/s]
 14%|█▍        | 56.0M/388M [00:00<00:03, 96.2MB/s]
 18%|█▊        | 68.0M/388M [00:00<00:03, 103MB/s] 
 22%|██▏       | 85.0M/388M [00:01<00:02, 122MB/s]
 25%|██▌       | 98.0M/388M [00:01<00:03, 85.7MB/s]
 28%|██▊       | 109M/388M [00:01<00:03, 89.8MB/s] 
 31%|███       | 120M/388M [00:01<00:03, 85.1MB/s]
 33%|███▎      | 130M/388M [00:01<00:03, 83.4MB/s]
 37%|███▋      | 142M/388M [00:01<00:02, 91.9MB/s]
 40%|███▉      | 154M/388M [00:01<00:02, 98.5MB/s]
 42%|████▏     | 165M/388M [00:01<00:02, 100MB/s] 
 46%|████▌     | 177M/388M [00:02<00:02, 105MB/s]
 49%|████▉     | 190M/388M [00:02<00:01, 111MB/s]
 52%|█████▏    | 201M/388M [00:02<00:01, 111MB/s]
 55%|█████▍    | 213M/388M [00:02<00:01, 99.4MB/s]
 59%|█████▉    | 229M/

✅ Google Perch v2 loaded successfully.


# [verification **test**](https://)

In [10]:
print(type(perch_model))

print("\nAvailable attributes:\n")

print(dir(perch_model))

<class 'perch_hoplite.zoo.taxonomy_model_tf.TaxonomyModelTF'>

Available attributes:

['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nonbatchable_batch_embed', 'batch_embed', 'batchable', 'class_list', 'embed', 'frame_audio', 'from_config', 'from_tfhub', 'get_classifier_head', 'hop_size_s', 'is_batchable', 'load_class_lists', 'load_surfperch_version', 'load_v2_version', 'load_version', 'model', 'model_path', 'normalize_audio', 'sample_rate', 'target_peak', 'tfhub_path', 'tfhub_version', 'window_size_s']


# **`Verify Processed Dataset Structure`**

In [11]:
import glob

BASE_DATASET_PATH = "/content/drive/MyDrive/PROCESSED_DATASET"

print("=== DATASET VERIFICATION ===")
for split in ["train", "validation", "test"]:
    for cls in ["panthera_leo", "background_ambient"]:
        path = f"{BASE_DATASET_PATH}/{split}/{cls}/*.wav"
        count = len(glob.glob(path))
        print(f"{split}/{cls} contains {count} WAV files")

=== DATASET VERIFICATION ===
train/panthera_leo contains 191 WAV files
train/background_ambient contains 92 WAV files
validation/panthera_leo contains 54 WAV files
validation/background_ambient contains 33 WAV files
test/panthera_leo contains 32 WAV files
test/background_ambient contains 22 WAV files


# **`Google Perch Embedding Extraction Pipeline`**

In [12]:
import numpy as np
from tqdm import tqdm

def extract_dataset_embeddings(base_dir="/content/drive/MyDrive/PROCESSED_DATASET"):
    # Storage containers
    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    # Split mapping
    splits = {
        "train": (X_train, y_train),
        "validation": (X_val, y_val),
        "test": (X_test, y_test)
    }

    # Binary labels
    classes = {
        "panthera_leo": 1,
        "background_ambient": 0
    }

    print("🚀 Starting Perch embedding extraction...")

    for split_name, (X_list, y_list) in splits.items():
        print(f"\nProcessing [{split_name.upper()}] split")
        for class_name, class_label in classes.items():
            folder_path = os.path.join(base_dir, split_name, class_name, "*.wav")
            audio_files = glob.glob(folder_path)

            print(f"Found {len(audio_files)} files for class [{class_name}]")

            for file_path in tqdm(audio_files, desc=f"{split_name}-{class_name}"):
                try:
                    # Load standardized audio
                    audio, _ = librosa.load(file_path, sr=32000, mono=True)

                    # Generate embeddings
                    outputs = perch_model.embed(audio.astype(np.float32))

                    # FIX: Flatten and isolate the precise 1,536 latent features natively
                    embedding_vector = np.array(outputs.embeddings).flatten()[:1536]

                    # Store embeddings
                    X_list.append(embedding_vector)
                    y_list.append(class_label)

                except Exception as e:
                    print(f"❌ Error processing: {file_path}")
                    print(e)
                    continue

    return (
        (np.array(X_train), np.array(y_train)),
        (np.array(X_val), np.array(y_val)),
        (np.array(X_test), np.array(y_test))
    )

# Execute extraction
(X_train, y_train), (X_val, y_val), (X_test, y_test) = extract_dataset_embeddings()

print("\n✅ Embedding extraction completed.")
print("\nTraining Shape:", X_train.shape)      # Should show: (283, 1536)
print("Validation Shape:", X_val.shape)  # Should show: (87, 1536)
print("Test Shape:", X_test.shape)          # Should show: (54, 1536)

🚀 Starting Perch embedding extraction...

Processing [TRAIN] split
Found 191 files for class [panthera_leo]


train-panthera_leo: 100%|██████████| 191/191 [08:12<00:00,  2.58s/it]


Found 92 files for class [background_ambient]


train-background_ambient: 100%|██████████| 92/92 [03:53<00:00,  2.53s/it]



Processing [VALIDATION] split
Found 54 files for class [panthera_leo]


...


Processing [TEST] split
Found 32 files for class [panthera_leo]


test-panthera_leo: 100%|██████████| 32/32 [01:20<00:00,  2.52s/it]


Found 22 files for class [background_ambient]


test-background_ambient: 100%|██████████| 22/22 [00:54<00:00,  2.48s/it]


✅ Embedding extraction completed.

Training Shape: (283, 1536)
Validation Shape: (87, 1536)
Test Shape: (54, 1536)


# **`Train Logistic Regression Classifier`**

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("Initializing classifier pipeline...")

lion_classifier_head = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            C=1.0,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42
        )
    )
])

print("🚀 Training classifier...")

lion_classifier_head.fit(
    X_train,
    y_train
)

print("✅ Logistic Regression classifier trained.")

Initializing classifier pipeline...
🚀 Training classifier...
✅ Logistic Regression classifier trained.


# **`VALIDATION`**

In [14]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    accuracy_score
)

# Validation predictions
val_predictions = lion_classifier_head.predict(X_val)

val_probabilities = (
    lion_classifier_head.predict_proba(X_val)[:, 1]
)

# Metrics
auc_score = roc_auc_score(
    y_val,
    val_probabilities
)

accuracy = accuracy_score(
    y_val,
    val_predictions
)

print("\n=== VALIDATION METRICS ===")

print(f"\nAccuracy Score: {accuracy:.4f}")

print(f"\nROC-AUC Score: {auc_score:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_val,
        val_predictions,
        target_names=[
            "Background Noise",
            "Lion"
        ]
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_val,
        val_predictions
    )
)

print("\n==========================")


=== VALIDATION METRICS ===

Accuracy Score: 0.9770

ROC-AUC Score: 0.9978

Classification Report:

                  precision    recall  f1-score   support

Background Noise       1.00      0.94      0.97        33
            Lion       0.96      1.00      0.98        54

        accuracy                           0.98        87
       macro avg       0.98      0.97      0.98        87
    weighted avg       0.98      0.98      0.98        87


Confusion Matrix:

[[31  2]
 [ 0 54]]



# **`SAVE MODEL`**

In [15]:
import joblib
import os

# Create model directory
os.makedirs(
    "/content/drive/MyDrive/MODELS",
    exist_ok=True
)

# Save classifier
joblib.dump(
    lion_classifier_head,
    "/content/drive/MyDrive/MODELS/lion_classifier.pkl"
)

print(
    "✅ Trained classifier saved successfully."
)

print(
    "Model path: "
    "/content/drive/MyDrive/MODELS/lion_classifier.pkl"
)

✅ Trained classifier saved successfully.
Model path: /content/drive/MyDrive/MODELS/lion_classifier.pkl


# **`the Standalone Edge Inference Script`**

In [16]:
%%writefile edge_inference.py
# ==============================================================================
# CYMASONIC LABS — SBM PROJECT: LIGHTWEIGHT EDGE INFERENCE RUNTIME
# Optimized for Raspberry Pi 5 Execution
# ==============================================================================

import os
import sys
import time
import numpy as np
import librosa
import joblib

# Suppress unnecessary warnings for clean terminal logging
import warnings
warnings.filterwarnings('ignore')

class EdgeLionDetector:
    def __init__(self, model_path="lion_classifier.pkl"):
        print(f"[{time.strftime('%H:%M:%S')}] Initializing Edge Inference Engine...")
        if not os.path.exists(model_path):
            print(f"❌ Error: Classifier model binary not found at {model_path}")
            sys.exit(1)

        # Load the lightweight scikit-learn classifier head pipeline
        self.model = joblib.load(model_path)
        print(f"[{time.strftime('%H:%M:%S')}] ✅ Custom Classification Head loaded successfully.")

        # Fixed constants matching Google Perch v2 native inputs
        self.TARGET_SR = 32000
        self.DURATION = 5
        self.REQUIRED_SAMPLES = self.TARGET_SR * self.DURATION # 160,000 samples

    def preprocess_buffer(self, raw_audio_array, current_sr):
        """
        Standardizes any incoming live audio array to 32kHz Mono, 5 seconds long,
        and normalizes amplitude peak to -3 dBFS.
        """
        # Force downmix to mono if stereo
        if len(raw_audio_array.shape) > 1:
            raw_audio_array = librosa.to_mono(raw_audio_array)

        # Resample to 32,000 Hz if necessary
        if current_sr != self.TARGET_SR:
            raw_audio_array = librosa.resample(raw_audio_array, orig_sr=current_sr, target_sr=self.TARGET_SR)

        # Enforce exact 5-second window via zero-padding or truncation
        if len(raw_audio_array) < self.REQUIRED_SAMPLES:
            raw_audio_array = np.pad(raw_audio_array, (0, self.REQUIRED_SAMPLES - len(raw_audio_array)), mode='constant')
        else:
            raw_audio_array = raw_audio_array[:self.REQUIRED_SAMPLES]

        # Peak Amplitude Normalization to -3 dBFS
        peak = np.max(np.abs(raw_audio_array))
        if peak > 1e-6:
            target_peak = 10 ** (-3 / 20)
            raw_audio_array = (raw_audio_array / peak) * target_peak

        return raw_audio_array.astype(np.float32)

    def run_telemetry_inference(self, clean_audio_buffer, raw_perch_model=None):
        """
        Runs the data buffer through the inference stack and measures latency.
        """
        start_time = time.time()

        # 1. Extract Embeddings
        # Note: In a production field box, raw_perch_model is called via tflite-runtime.
        # For our baseline test, we simulate extracting the 1,536-D vector.
        if raw_perch_model is not None:
            outputs = raw_perch_model.embed(clean_audio_buffer)
            embedding_vector = np.array(outputs.embeddings).flatten()[:1536]
        else:
            # Simulation fallback if testing framework without model loading
            embedding_vector = np.zeros((1536,))

        # 2. Predict using the lightweight linear probe head
        prediction = self.model.predict([embedding_vector])[0]
        probability = self.model.predict_proba([embedding_vector])[0][1]

        latency = time.time() - start_time

        # 3. Compile Telemetry Log
        status = "🚨 LION DETECTED" if prediction == 1 else "🌲 BACKGROUND AMBIENT"
        print(f"[{time.strftime('%H:%M:%S')}] Status: {status:<20} | Confidence: {probability*100:6.2f}% | Latency: {latency*1000:4.0f}ms")

        return prediction, probability, latency

# Example Execution Block if run directly
if __name__ == "__main__":
    detector = EdgeLionDetector(model_path="/content/drive/MyDrive/MODELS/lion_classifier.pkl")
    print("Edge device script compiled and waiting for live microphone buffers.")

Writing edge_inference.py


# **`Add the Live Microphone Recorder Cell`**

In [17]:
# ==============================================================================
# LIVE MICROPHONE RECORDING INTERFACE (BROWSER-BASED)
# ==============================================================================
from IPython.display import display, Javascript
from google.colab import output
import base64

RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    blob = new Blob(chunks, { type: 'audio/wav' });
    text = await b2text(blob);
    resolve(text);
  };
  recorder.stop();
});
"""

def record_live_audio(seconds=5):
    """Activates your laptop microphone to record a live 5-second window."""
    print(f"🎤 SYSTEM READY: Prepare your phone sound... Recording starts in 1 second.")
    import time
    time.sleep(1)
    print("🔴 RECORDING NOW... Play the sound directly into the microphone!")
    display(Javascript(RECORD_JS))
    s = output.eval_js('record(%d)' % (seconds * 1000))
    print("🛑 RECORDING COMPLETE. Processing audio buffer...")
    b = base64.b64decode(s.split(',')[1])

    # Save the live recorded clip temporarily
    with open("live_test.wav", "wb") as f:
        f.write(b)
    return "live_test.wav"

In [20]:
# Reload bird taxonomy labels

labels = perch_model.class_list["labels"]

bird_classes = labels.classes

print(
    f"✅ Bird taxonomy loaded: {len(bird_classes)} species"
)

✅ Bird taxonomy loaded: 14795 species


# `Add the Live Inference Trigger Cell`

In [45]:
import tensorflow as tf
import librosa
import numpy as np

# ============================================
# RECORD LIVE AUDIO
# ============================================

audio_file_path = record_live_audio(seconds=5)

# ============================================
# LOAD AUDIO
# ============================================

audio, _ = librosa.load(
    audio_file_path,
    sr=32000,
    mono=True
)

# Force exact 5 sec
required_samples = 32000 * 5

if len(audio) < required_samples:

    audio = np.pad(
        audio,
        (0, required_samples - len(audio))
    )

else:

    audio = audio[:required_samples]

# ============================================
# PREPARE INPUT
# ============================================

audio_batch = np.expand_dims(
    audio.astype(np.float32),
    axis=0
)

# ============================================
# PERCH FULL INFERENCE
# ============================================

outputs = perch_model.model.signatures[
    "serving_default"
](
    inputs=tf.constant(audio_batch)
)

# ============================================
# EXTRACT EMBEDDINGS
# ============================================

embedding_vector = outputs[
    "embedding"
].numpy()[0]

embedding_vector = embedding_vector.reshape(1, -1)

# ============================================
# RUN LION CLASSIFIER
# ============================================

lion_prediction = lion_classifier_head.predict(
    embedding_vector
)[0]

lion_probabilities = lion_classifier_head.predict_proba(
    embedding_vector
)[0]

lion_confidence = lion_probabilities[1]

# ============================================
# RUN PERCH BIRD CLASSIFIER
# ============================================

bird_logits = outputs["label"].numpy()[0]

# Apply softmax to convert logits to probabilities
bird_probabilities = tf.nn.softmax(bird_logits).numpy()

bird_index = np.argmax(bird_probabilities)

bird_confidence = bird_probabilities[bird_index]

bird_species = bird_classes[bird_index]

# ============================================
# FINAL DECISION ENGINE
# ============================================

print("\n" + "="*50)
print("📡 CYMASONIC UNIVERSAL BIOACOUSTIC ENGINE")
print("="*50)

# PRIORITY 1 → LION
if lion_confidence > 0.80:

    print("\n🚨 LION DETECTED")

    print(f"\nLion Confidence: {lion_confidence:.4f}")

# PRIORITY 2 → BIRD
elif bird_confidence > 0.25:

    print("\n🚨 GOOGLE PERCH DETECTED")

    print(f"\nSpecies: {bird_species}")

    print(f"\nBird Confidence: {bird_confidence:.4f}")

# OTHERWISE → BACKGROUND
else:

    print("\n🌲 BACKGROUND NOISE")

    print(
        "\nNo strong lion or bird signature detected."
    )

print("\n" + "="*50)

🎤 SYSTEM READY: Prepare your phone sound... Recording starts in 1 second.
🔴 RECORDING NOW... Play the sound directly into the microphone!


<IPython.core.display.Javascript object>

🛑 RECORDING COMPLETE. Processing audio buffer...


/tmp/ipykernel_4247/3239746008.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



📡 CYMASONIC UNIVERSAL BIOACOUSTIC ENGINE

🚨 GOOGLE PERCH DETECTED

Species: Corvus brachyrhynchos

Bird Confidence: 0.3362



# **`ONLY FOR BIRDS TEST `**

In [ ]:
# Record live microphone audio
audio_file_path = record_live_audio(seconds=5)

print(audio_file_path)

import tensorflow as tf
import librosa
import numpy as np

# Load live recorded audio
audio, _ = librosa.load(
    audio_file_path,
    sr=32000,
    mono=True
)

# Ensure exactly 5 sec
required_samples = 32000 * 5

if len(audio) < required_samples:
    audio = np.pad(
        audio,
        (0, required_samples - len(audio))
    )
else:
    audio = audio[:required_samples]

# Add batch dimension
audio_batch = np.expand_dims(
    audio.astype(np.float32),
    axis=0
)

# Run Perch native classifier
outputs = perch_model.model.signatures[
    "serving_default"
](
    inputs=tf.constant(audio_batch)
)

# Extract bird logits
bird_logits = outputs["label"].numpy()[0]

# Top prediction
top_index = np.argmax(bird_logits)

# Species name
bird_species = bird_classes[top_index]

# Confidence
confidence = bird_logits[top_index]

print("\n=== PERCH LIVE BIRD PREDICTION ===")

print(f"\nPredicted Species: {bird_species}")

print(f"\nConfidence: {confidence:.4f}")

# Task
Review and evaluate the provided bioacoustic engine pipeline, which includes lion detection using Google Perch embeddings and a logistic regression classifier, and bird detection using Google Perch's native classification. The review will cover data processing, embedding extraction, model training, validation, and deployment readiness of the edge inference script. Finally, an overall score and justification will be provided.

## Review Lion Detection Pipeline

### Subtask:
Analyze the data processing, embedding extraction, and logistic regression training to confirm the pipeline's effectiveness in detecting lion voices based on your provided data. This includes examining the `run_dsp_pipeline`, `process_dataset`, embedding extraction in `extract_dataset_embeddings`, and the training/validation of `lion_classifier_head`.


**Reasoning**:
The first step is to examine the `run_dsp_pipeline` function as per the instructions. I will display the code of the function using the `code_block` command.



In [27]:
import os
import librosa
import soundfile as sf
import numpy as np

def run_dsp_pipeline(source_file_path, destination_folder):
    TARGET_SR = 32000
    SEGMENT_DURATION = 5
    SAMPLES_PER_SEGMENT = TARGET_SR * SEGMENT_DURATION

    try:
        # Load audio, auto-enforcing mono array conversion at 32kHz
        audio_signal, sr = librosa.load(source_file_path, sr=TARGET_SR, mono=True)

        peak_amplitude = np.max(np.abs(audio_signal))

        # Drop corrupted or silent data files to maintain vector integrity
        if peak_amplitude < 1e-6:
            print(f"Skipping silent file: {source_file_path}")
            return 0

        # Mathematical peak normalization to -3 dBFS
        target_peak = 10 ** (-3 / 20)
        audio_signal = (audio_signal / peak_amplitude) * target_peak

        base_name = os.path.splitext(os.path.basename(source_file_path))[0]
        total_samples = len(audio_signal)
        chunk_count = 0

        # Slice long timelines into uniform 5-second arrays (160,000 values)
        for start_sample in range(0, total_samples, SAMPLES_PER_SEGMENT):
            end_sample = start_sample + SAMPLES_PER_SEGMENT
            audio_chunk = audio_signal[start_sample:end_sample]

            # Zero-padding: pads short trailing clips with silent numbers
            if len(audio_chunk) < SAMPLES_PER_SEGMENT:
                audio_chunk = np.pad(
                    audio_chunk,
                    (0, SAMPLES_PER_SEGMENT - len(audio_chunk)),
                    mode='constant'
                )

            output_filename = f"{base_name}_chunk_{chunk_count}.wav"
            output_path = os.path.join(destination_folder, output_filename)

            # Export clean WAV segment
            sf.write(output_path, audio_chunk, TARGET_SR)
            chunk_count += 1

        return chunk_count

    except Exception as e:
        print(f"❌ Error processing {source_file_path}: {e}")
        return 0

```markdown
### Analysis of `run_dsp_pipeline` function:

The `run_dsp_pipeline` function is responsible for preprocessing audio files. Here's a breakdown of its functionality:

**Strengths:**
*   **Standardized Resampling and Mono Conversion**: Audio files are consistently loaded at a `TARGET_SR` of 32kHz and converted to mono, which is crucial for uniform processing and compatibility with the Google Perch model.
*   **Silence Handling**: It includes a mechanism to skip silent files (`peak_amplitude < 1e-6`), preventing the processing of empty or corrupted audio, which helps maintain data integrity.
*   **Peak Normalization**: Mathematical peak normalization to -3 dBFS ensures that audio segments have a consistent amplitude level, which can be important for model stability and performance.
*   **Segmentation**: The function effectively slices long audio files into uniform 5-second segments (`SEGMENT_DURATION = 5`), a standard practice for many audio classification tasks, especially for models like Google Perch.
*   **Zero-Padding**: Short trailing clips are zero-padded to meet the `SAMPLES_PER_SEGMENT` requirement, ensuring all segments have the exact same length. This is critical for fixed-input models.
*   **Error Handling**: The `try-except` block provides basic error handling for individual file processing, preventing the entire pipeline from crashing due to a single problematic file.
*   **Clear Output Structure**: The output files are named systematically (`base_name_chunk_chunk_count.wav`), making them easy to identify and manage.

**Potential Improvements/Considerations:**
*   **Peak Amplitude Threshold**: The `1e-6` threshold for silent files might be too strict or too lenient depending on the audio characteristics. It could be made configurable or adjusted based on empirical analysis of the dataset.
*   **Error Logging**: While it prints an error, a more robust logging mechanism (e.g., writing to a log file, including timestamps and more details) would be beneficial for debugging and monitoring in a production environment.
*   **Parallel Processing**: For large datasets, processing files sequentially can be slow. Integrating parallel processing (e.g., using `multiprocessing` or `joblib`) could significantly speed up the data preparation phase.
*   **Destination Folder**: The `destination_folder` is passed as an argument. Ensuring that the `process_dataset` function correctly creates these destination folders before calling `run_dsp_pipeline` is important to avoid errors (which is handled by `os.makedirs(directory, exist_ok=True)` in `process_dataset`).

Overall, the `run_dsp_pipeline` function is well-structured and implements essential audio preprocessing steps effectively for the given task.

# Task
Review and evaluate the provided bioacoustic engine pipeline, which includes lion detection using Google Perch embeddings and a logistic regression classifier, and bird detection using Google Perch's native classification. The review will cover data processing, embedding extraction, model training, validation, and deployment readiness of the edge inference script. Finally, an overall score and justification will be provided.

## Modify Bird Confidence Calculation

### Subtask:
Modify cell `5j3E4eBlN17F` to apply `tf.nn.softmax` to `bird_logits` to convert them into actual probabilities. This will ensure that `bird_confidence` is a value between 0 and 1, making the `0.25` threshold meaningful for bird detection.
